# Aggregation and Grouping

*Summarize a table, calculate grouped revenue, and distinguish row filters from group filters.*


## Prepare the Sales Table

The sales table holds five prepared records in an in-memory
SQLite database. `sale_id` is a primary key, and the other fields store
the product, category, quantity, whole-dollar unit price, and month number.
`execute()` runs one SQL statement; `executemany()` binds each prepared
tuple to the `?` placeholders. `commit()` completes the insert transaction.


In [1]:
import sqlite3

connection = sqlite3.connect(":memory:")

connection.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price INTEGER NOT NULL,
    month INTEGER NOT NULL
)
""")

sales_rows = [
    ("Mouse", "Accessory", 3, 25, 1),
    ("Keyboard", "Accessory", 2, 45, 1),
    ("Monitor", "Display", 1, 180, 2),
    ("Laptop Stand", "Accessory", 4, 30, 2),
    ("Tablet", "Device", 2, 220, 3),
]

connection.executemany("""
INSERT INTO sales (product_name, category, quantity, unit_price, month)
VALUES (?, ?, ?, ?, ?)
""", sales_rows)
connection.commit()

print("Prepared sales:")
for row in connection.execute("SELECT * FROM sales ORDER BY sale_id"):
    print(row)


Prepared sales:
(1, 'Mouse', 'Accessory', 3, 25, 1)
(2, 'Keyboard', 'Accessory', 2, 45, 1)
(3, 'Monitor', 'Display', 1, 180, 2)
(4, 'Laptop Stand', 'Accessory', 4, 30, 2)
(5, 'Tablet', 'Device', 2, 220, 3)


There are five sales, numbered 1 through 5. Mouse, Keyboard, and
Laptop Stand belong to Accessory; Monitor belongs to Display,
and Tablet belongs to Device.


## Aggregate the Whole Table

`COUNT(*)` counts rows. `SUM`, `AVG`, `MIN`, and `MAX` summarize a numeric
expression. SQL uses `NULL` for missing or unknown values, not zero.
`COUNT(column)` counts only non-NULL values; the other listed
numeric aggregates also ignore NULL inputs. The prepared fields are all known.
`AS` names each result for interpretation.


In [2]:
summary = connection.execute("""
SELECT COUNT(*) AS sale_count,
       SUM(quantity) AS total_quantity,
       AVG(unit_price) AS average_price,
       MIN(unit_price) AS lowest_price,
       MAX(unit_price) AS highest_price,
       SUM(quantity * unit_price) AS total_revenue
FROM sales
""").fetchone()

labels = ["Sale count", "Total quantity", "Average unit price", "Lowest price", "Highest price", "Total revenue"]
for label, value in zip(labels, summary):
    print(f"{label}: {value}")


Sale count: 5
Total quantity: 12
Average unit price: 100.0
Lowest price: 25
Highest price: 220
Total revenue: 905


Five sales contain 12 items and total 905 dollars. The mean of the five unit prices is 100 dollars; it is not a quantity-weighted mean. Prices range from 25 to 220.


## Group Records before Summarizing

`GROUP BY category` produces one summary row per category. Every selected
field here is either a group key or an aggregate, so its meaning is clear.
`ORDER BY` defines the display order independently of grouping.


In [3]:
print("Category, sale count, quantity, and revenue:")
for row in connection.execute("""
SELECT category,
       COUNT(*) AS sale_count,
       SUM(quantity) AS total_quantity,
       SUM(quantity * unit_price) AS total_revenue
FROM sales
GROUP BY category
ORDER BY total_revenue DESC, category ASC
"""):
    print(row)


Category, sale count, quantity, and revenue:
('Device', 1, 2, 440)
('Accessory', 3, 9, 285)
('Display', 1, 1, 180)


Device has the largest revenue, 440, from one Tablet sale record. Accessory totals 285 across three records, and Display totals 180 from one record.


## Filter Rows with `WHERE` and Groups with `HAVING`

`WHERE` filters individual rows before aggregation. `HAVING` filters
grouped results after aggregation. This query first restricts records to
months 1 and 2, then keeps categories whose retained revenue is at least 250.


In [4]:
print("Categories reaching 250 dollars in months 1 and 2:")
for row in connection.execute("""
SELECT category, SUM(quantity * unit_price) AS total_revenue
FROM sales
WHERE month <= ?
GROUP BY category
HAVING SUM(quantity * unit_price) >= ?
ORDER BY total_revenue DESC, category ASC
""", (2, 250)):
    print(row)


Categories reaching 250 dollars in months 1 and 2:
('Accessory', 285)


Only Accessory remains, at 285 dollars. Device's month-3 record was removed by `WHERE`; Display's 180-dollar group fails `HAVING`.


## Close the Connection

`.close()` releases the database connection. This database lives only
in memory, so its tables disappear when the connection closes.


In [5]:
connection.close()
print("Connection closed.")


Connection closed.


Use aggregates to state the measure, `GROUP BY` to state its level of detail, `WHERE` to choose input rows, and `HAVING` to choose output groups.

[SQLite aggregate functions](https://www.sqlite.org/lang_aggfunc.html)
